# MT5 historical data fetch

This notebook fetches historical OHLCV candles directly from the MetaTrader 5 terminal.

- Configure symbol, timeframe, and date range (or days back)
- Uses the same output JSON format and naming convention as `scripts/fetch_data.py`
- Saves to the repository `data/` folder: `data/candle_data_list_frxSYMBOL_SECONDSs.json`

Requirements:
- MetaTrader 5 desktop terminal running and authorized
- Python `MetaTrader5` package installed for the interpreter running this notebook
- Correct broker symbol (e.g., `EURUSDm`). If you pass a generic symbol (e.g., `frxEURUSD`), the code will try a few common alias variations.


In [1]:
# Configuration: symbol, timeframe, and date range
import os
import json
from datetime import datetime, timedelta, timezone
from pathlib import Path
import pandas as pd
try:
    import yaml
except Exception:
    yaml = None

# Load defaults from trading bot config if available
default_symbol = "frxEURUSD"
default_timeframe = "5m"
login = None
password = None
server = None
cfg_path = Path("configs/trading_bot_config.yaml")
if cfg_path.exists() and yaml is not None:
    try:
        with cfg_path.open("r", encoding="utf-8") as f:
            bot_cfg = yaml.safe_load(f)
        default_symbol = bot_cfg.get("general", {}).get("default_symbol", default_symbol)
        default_timeframe = bot_cfg.get("general", {}).get("default_timeframe", default_timeframe)
        acc = bot_cfg.get("account", {})
        login = acc.get("login")
        password = acc.get("password")
        server = acc.get("server")
    except Exception as e:
        print("Warning: failed to read trading bot config:", e)
        
# Parameters you can change
symbol = default_symbol           # e.g., 'EURUSDm' for broker symbol, or 'frxEURUSD' (alias)
timeframe = default_timeframe     # one of: '1m','5m','15m','30m','1h','4h','1d'
days_back = 7                     # alternatively, set a range below and set days_back=None
startdate = None                  # e.g., '2025-11-25'
enddate = None                    # e.g., '2025-12-10'

.strip()

SyntaxError: invalid syntax (3233148393.py, line 39)

In [ ]:
# Fetch data from MetaTrader 5
import sys
try:
    import MetaTrader5 as mt5
except Exception as e:
    raise RuntimeError(f"Failed to import MetaTrader5: {e}")

def _tf_map(mt5_module):
    return {
        '1m': mt5_module.TIMEFRAME_M1,
        '5m': mt5_module.TIMEFRAME_M5,
        '15m': mt5_module.TIMEFRAME_M15,
        '30m': mt5_module.TIMEFRAME_M30,
        '1h': mt5_module.TIMEFRAME_H1,
        '4h': mt5_module.TIMEFRAME_H4,
        '1d': mt5_module.TIMEFRAME_D1,
    }

def _select_symbol_with_aliases(sym: str) -> str:
    """Try common alias variations to select the broker symbol, return selected name."""
    candidates = []
    s = sym
    candidates.append(s)
    # Strip frx prefix
    if s.lower().startswith('frx'):
        candidates.append(s[3:])
    # Strip trailing 'm' (common suffix for Exness)
    if s.endswith('m'):
        candidates.append(s[:-1])
    # Both frx and m
    if s.lower().startswith('frx') and s.endswith('m'):
        candidates.append(s[3:-1])
    tried = set()
    for c in candidates:
        if c in tried or not c:
            continue
        tried.add(c)
        if mt5.symbol_select(c, True):
            return c
    return sym  # fallback

# Initialize and optional login
init_ok = mt5.initialize()
print("mt5.initialize() ->", init_ok)
print("mt5.last_error() ->", mt5.last_error())
if not init_ok:
    raise RuntimeError(f"MT5 initialize failed: {mt5.last_error()}")

if login and password and server:
    try:
        authorized = mt5.login(int(login), password=str(password), server=str(server))
        print("mt5.login() ->", authorized)
        print("mt5.last_error() after login ->", mt5.last_error())
    except Exception as e:
        print("mt5.login exception:", e)
else:
    print("Skipping mt5.login(): no credentials provided (using current terminal session)")

tf_map = _tf_map(mt5)
if timeframe not in tf_map:
    raise ValueError(f"Unsupported timeframe: {timeframe}")

# Build date range
if startdate and enddate:
    start_dt = pd.to_datetime(startdate, utc=True)
    end_dt = pd.to_datetime(enddate, utc=True)
elif days_back is not None:
    end_dt = datetime.now(timezone.utc).replace(second=0, microsecond=0)
    start_dt = end_dt - timedelta(days=int(days_back))
else:
    raise ValueError("Provide either days_back or both startdate and enddate")

selected_symbol = _select_symbol_with_aliases(symbol)
print(f"Selected MT5 symbol: {selected_symbol}")

rates = mt5.copy_rates_range(selected_symbol, tf_map[timeframe], start_dt, end_dt)
if rates is None or len(rates) == 0:
    raise RuntimeError(f"Failed to fetch rates for {selected_symbol}: {mt5.last_error()}")

df = pd.DataFrame(rates)
df['time'] = pd.to_datetime(df['time'], unit='s', utc=True)
df = df.rename(columns={'time': 'Date', 'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 'tick_volume': 'Volume'})
df = df[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].set_index('Date').sort_index()
print(f"Fetched {len(df)} rows from MT5 for {selected_symbol} {timeframe}")
.strip()

In [ ]:
# Save to data/ in the expected JSON format and naming
import math

def _timeframe_to_seconds(tf: str) -> int:
    tf = tf.lower().strip()
    if tf.endswith('m'):
        return int(tf[:-1]) * 60
    elif tf.endswith('h'):
        return int(tf[:-1]) * 3600
    elif tf.endswith('d'):
        return int(tf[:-1]) * 86400
    else:
        raise ValueError(f"Unsupported timeframe: {tf}")

# Build output records with Date in 'YYYY-MM-DD HH:MM:SS+00:00' format
records = []
for idx, row in df.reset_index().iterrows():
    date_str = row['Date'].isoformat().replace('T', ' ')  # ensure space instead of 'T'
    records.append({
        'Date': date_str,
        'Open': float(row['Open']),
        'High': float(row['High']),
        'Low': float(row['Low']),
        'Close': float(row['Close']),
        'Volume': float(row['Volume'])
    })

# Determine output filename consistent with scripts/fetch_data.py
symbol_for_filename = f"frx{symbol.upper()}" if not symbol.lower().startswith('frx') else symbol.upper()
timeframe_seconds = _timeframe_to_seconds(timeframe)
out_dir = Path('data')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"candle_data_list_{symbol_for_filename}_{timeframe_seconds}s.json"

with out_path.open('w', encoding='utf-8') as f:
    json.dump(records, f, indent=2)

print(f"Saved {len(records)} candles to {out_path}")
print(f"Date range: {records[0]['Date']} .. {records[-1]['Date']}")
.strip()

In [ ]:
# Preview a few rows
df.head(5)